In [1]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
from collections import Counter
import anndata as ad
from scipy import sparse
from scipy.io import mmwrite


workdir = "./ComicGTN_reproducibility/Data/Mouse_retina"
RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
ATAC_seq = sc.read(os.path.join(workdir, "Peak_Cell.mtx"))
Cell_names = pd.read_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", header = None)
Cell_types = pd.read_csv(os.path.join(workdir, "Cell_types.tsv"), sep = "\t", header = None)
Counter(Cell_types[0])
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
RNA_seq.obs_names = Gene_names[0]
RNA_seq.var_names = Cell_names[0]
ATAC_seq.obs_names = Peak_names[0]
ATAC_seq.var_names = Cell_names[0]
RNA_count = RNA_seq.X 
ATAC_count = ATAC_seq.X


Cell_types.columns = ["cell_type"]
Cell_types.index = Cell_names[0]
Cell_types["number"] = 1
Cell_count = Cell_types.groupby("cell_type").count()
Cell_count =Cell_count.sort_values(by = "number",ascending = True, inplace = False)
true_label = Cell_types.loc[Cell_names[0].tolist(), :]["cell_type"]


def Sample_Anndatas(index, ad1, dim1, ad2, dim2):
    
    def sub_ad(ad, id, dim):
        if dim == 0:
            ad = ad[id, :]
        elif dim == 1:
            ad = ad[:, id]
            
        return(ad)

    ad1 = sub_ad(ad1, index, dim1)
    ad2 = sub_ad(ad2, index, dim2)

    # 删除在当前细胞中不活跃的 gene 和 peak
    index_genes = np.where(np.array(ad1.X.sum(dim1)).squeeze() > 0)[0]
    index_peaks = np.where(np.array(ad2.X.sum(dim2)).squeeze() > 0)[0]
    ad1 = sub_ad(ad1, index_genes, 1 - dim1)
    ad2 = sub_ad(ad2, index_peaks, 1 - dim2)
    
    # 删除在当前 gene 和 peak 中不表达的细胞
    index_cells = np.where((np.array(ad1.X.sum(1 - dim1)).squeeze() > 0) & 
                                         (np.array(ad2.X.sum(1 - dim2)).squeeze() > 0))[0]
    ad1 = sub_ad(ad1, index_cells, dim1)
    ad2 = sub_ad(ad2, index_cells, dim2)
    
    return (ad1, ad2, index_genes, index_peaks, index_cells)

In [6]:
import random
rare_type = ["BC10", "HC"]
ordinary_type = ["BC1"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [248, 30, 22]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim0_Ordinary1_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

/tmp/ipykernel_1852844/779930368.py:14: DeprecationWarning: Sampling from a set deprecated
since Python 3.9 and will be removed in a subsequent version.
  l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])


In [2]:
# 基于 Mouse_retina 的模拟数据 Sim1，包含 500 个细胞
rare_type = ["BC9"]
ordinary_type = ["rod"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [490, 10]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim1_Ordinary1_Rare1"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [3]:
# 基于 Mouse_retina 的模拟数据 Sim2，包含 785 个细胞
rare_type = ["BC7", "BC8"]
ordinary_type = ["rod"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [755, 15, 15]

for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim2_Ordinary1_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [4]:
# 基于 Mouse_retina 的模拟数据 Sim3，包含 1200 个细胞
rare_type = ["BC7", "BC10"]
ordinary_type = ["rod", "MG"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [860, 300, 20, 20]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])
            

l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim3_Ordinary2_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [5]:
# 基于 Mouse_retina 的模拟数据 Sim4，包含 1840 个细胞
rare_type = ["AC", "BC6"]
ordinary_type = ["rod", "RGC", "Cone"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1002, 402, 400, 18, 18]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

        
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim4_Ordinary3_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [6]:
# 基于 Mouse_retina 的模拟数据 Sim5，包含 2450 个细胞
rare_type = ["BC8", "BC9", "HC"]
ordinary_type = ["rod", "MG", "Cone"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1610, 400, 380, 30, 20, 10]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

        
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim5_Ordinary3_Rare3"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [7]:
# 基于 Mouse_retina 的模拟数据 Sim6，包含 3000 个细胞
rare_type = ["BC5", "AC", "BC6"]
ordinary_type = ["rod", "MG", "RGC", "Cone"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [2000, 300, 300, 300, 40, 30, 30]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

        
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim6_Ordinary4_Rare3"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [8]:
# 基于 Mouse_retina 的模拟数据 Sim7，包含 3940 个细胞
rare_type = ["BC8", "BC9"]
ordinary_type = ["rod", "MG", "RGC", "Cone", "BC1"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [2600, 400, 400, 300, 200, 20, 20]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

        
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim7_Ordinary5_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [9]:
# 基于 Mouse_retina 的模拟数据 Sim8，包含 4875 个细胞
rare_type = ["BC9", "BC10", "HC"]
ordinary_type = ["rod", "MG", "RGC", "Cone", "BC1", "BC2"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [3200, 400, 380, 350, 320, 160, 35, 20, 10]


for i in range(0, total_num):
        l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

        
l = np.sort(l)
label = true_label[l]
Sim_RNA_count, Sim_ATAC_count, gene_index, peak_index, cell_index = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sim_RNA_count.X
ATAC_matrix = Sim_ATAC_count.X
cell_name_df  = pd.DataFrame(list(Cell_names[0][cell_index]), columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim8_Ordinary6_Rare3"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)

In [12]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.io import mmwrite


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell_merge.mtx"))
Cell_names = pd.read_csv(os.path.join(workdir, "Cell_names_merge.tsv"), sep = "\t", header = None)
Cell_types = pd.read_csv(os.path.join(workdir, "Cell_types_merge.tsv"), sep = "\t", header = None)
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names_merge.tsv"), sep = "\t", header = None)
RNA_seq.obs_names = Gene_names[0]
RNA_seq.var_names = Cell_names[0]
RNA_count = RNA_seq.X 


Cell_types.columns = ["cell_type"]
Cell_types.index = Cell_names[0]
Cell_types["number"] = 1
Cell_count = Cell_types.groupby("cell_type").count()
Cell_count =Cell_count.sort_values(by = "number",ascending = True, inplace = False)
true_label = Cell_types.loc[Cell_names[0].tolist(), :]["cell_type"]


def Sample_Anndatas(index, ad1, dim1):
    
    def sub_ad(ad, id, dim):
        if dim == 0:
            ad = ad[id, :]
        elif dim == 1:
            ad = ad[:, id]
            
        return(ad)

    ad1 = sub_ad(ad1, index, dim1)

    # 删除在当前细胞中不活跃的 gene 和 peak
    index_genes = np.where(np.array(ad1.X.sum(dim1)).squeeze() > 0)[0]
    ad1 = sub_ad(ad1, index_genes, 1 - dim1)
    
    # 删除在当前 gene 和 peak 中不表达的细胞
    index_cells = np.where((np.array(ad1.X.sum(1 - dim1)).squeeze() > 0))[0]
    ad1 = sub_ad(ad1, index_cells, dim1)
    
    return (ad1, index_genes, index[index_cells])

In [11]:
# 基于 BMMC-bench-3 的模拟数据 Sim9，包含 1425 个细胞
rare_type = ["CD8+ T naive", "Normoblast"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [575, 475, 350, 13, 12]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim9_Ordinary3_Rare2"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim9_Ordinary3_Rare2"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "CD8+ T naive": 2, "Normoblast": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [12]:
# 基于 BMMC-bench-3 的模拟数据 Sim10，包含 2362 个细胞
rare_type = ["cDC2", "Normoblast", "pDC"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [720, 560, 530, 490, 28, 18, 16]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])
            

l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim10_Ordinary4_Rare3"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim10_Ordinary4_Rare3"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, 
                                                                                     "cDC2": 2, "Normoblast": 2, "pDC": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [13]:
# 基于 BMMC-bench-3 的模拟数据 Sim11，包含 3475 个细胞
rare_type = ["HSC", "cDC2", "pDC", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1020, 700, 570, 550, 530, 40, 35, 18, 12]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim11_Ordinary5_Rare4"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim11_Ordinary5_Rare4"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "HSC": 2, "cDC2": 2, "pDC": 2, "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [14]:
# 基于 BMMC-bench-3 的模拟数据 Sim12，包含 4777 个细胞
rare_type = ["G/M prog", "Normoblast", "pDC", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1240, 880, 740, 620, 555, 545, 87, 42, 36, 32]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim12_Ordinary6_Rare4"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim12_Ordinary6_Rare4"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1,
                                                                                     "G/M prog": 2, "Normoblast": 2, "pDC": 2, "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [15]:
# 基于 BMMC-bench-3 的模拟数据 Sim13，包含 5432 个细胞
rare_type = ["MK/E prog", "HSC", "cDC2", "CD8+ T naive", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1210, 850, 800, 690, 620, 520, 430, 98, 69, 60, 48, 37]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim13_Ordinary7_Rare5"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim13_Ordinary7_Rare5"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, 
                                                                                     "MK/E prog": 2, "HSC": 2, "cDC2": 2, "CD8+ T naive": 2, "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [16]:
# 基于 BMMC-bench-3 的模拟数据 Sim14，包含 6388 个细胞
rare_type = ["MK/E prog", "G/M prog", "cDC2", "Normoblast", "pDC"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast", "Lymph prog"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1470, 1005, 805, 735, 615, 500, 500, 420, 106, 85, 68, 44, 35]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim14_Ordinary8_Rare5"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim14_Ordinary8_Rare5"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, "Lymph prog": 1,
                                                                                     "MK/E prog": 2, "G/M prog": 2, "cDC2": 2, "Normoblast": 2, "pDC": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [17]:
# 基于 BMMC-bench-3 的模拟数据 Sim15，包含 8124 个细胞
rare_type = ["CD16+ Mono", "G/M prog", "HSC", "Normoblast", "pDC", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast", 
                            "Lymph prog", "B1 B"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1670, 1235, 925, 750, 740, 680, 600, 500, 465, 160, 122, 104, 74, 55, 44]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim15_Ordinary9_Rare6"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim15_Ordinary9_Rare6"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, "Lymph prog": 1, "B1 B": 1,
                                                                                     "CD16+ Mono": 2, "G/M prog": 2, "HSC": 2, "Normoblast": 2, "pDC": 2, 
                                                                                     "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [18]:
# 基于 BMMC 的模拟数据 Sim16，包含 9231 个细胞
rare_type = ["G/M prog", "HSC", "cDC2", "CD8+ T naive", "Normoblast", "pDC"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast",
                            "Lymph prog", "B1 B", "Erythroblast"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1540, 1140, 970, 880, 880, 780, 700, 650, 550, 480, 165, 153, 120, 89, 78, 56]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim16_Ordinary10_Rare6"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim16_Ordinary10_Rare6"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1,
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, "Lymph prog": 1, "B1 B": 1, "Erythroblast": 1,
                                                                                     "G/M prog": 2, "HSC": 2, "cDC2": 2, "CD8+ T naive": 2, "Normoblast": 2, "pDC": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [19]:
# 基于 BMMC 的模拟数据 Sim17，包含 10110 个细胞
rare_type = ["CD16+ Mono", "MK/E prog", "cDC2", "CD8+ T naive", "Normoblast", "pDC", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast", 
                            "Lymph prog", "B1 B", "Erythroblast"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1700, 1200, 1125, 900, 900, 800, 700, 650, 650, 630, 210, 180, 130, 100, 90, 80, 65]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim17_Ordinary10_Rare7"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim17_Ordinary10_Rare7"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, "Lymph prog": 1, "B1 B": 1, "Erythroblast": 1,
                                                                                     "CD16+ Mono": 2, "MK/E prog": 2, "cDC2": 2, "CD8+ T naive": 2, "Normoblast": 2,
                                                                                     "pDC": 2, "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)

In [20]:
# 基于 BMMC 的模拟数据 Sim18，包含 12336 个细胞
rare_type = ["CD16+ Mono", "G/M prog", "HSC", "cDC2", "Normoblast", "pDC", "Plasma cell"]
ordinary_type = ["CD8+ T", "Naive CD20+ B", "NK", "Transitional B", "CD14+ Mono", "CD4+ T activated", "Proerythroblast", 
                            "Lymph prog", "B1 B", "Erythroblast", "CD4+ T naive"]
type_choice = ordinary_type + rare_type
total_num = len(type_choice)


name_list = type_choice
l = list()
num_list = [1860, 1450, 1280, 1120, 1000, 900, 810, 740, 740, 690, 670, 256, 194, 185, 166, 115, 96, 64]


for i in range(0, total_num):
    l = l + random.sample(set(true_label[true_label == type_choice[i]].index), num_list[i])

    
l = np.sort(l)
label = true_label[l]
Sam_RNA_count, Sam_ATAC_count, gene_index, peak_index, cell_names = Sample_Anndatas(l, RNA_seq, 1, ATAC_seq, 1)
RNA_matrix = Sam_RNA_count.X
ATAC_matrix = Sam_ATAC_count.X
cell_name_df  = pd.DataFrame(cell_names, columns = [0])
gene_name_df = pd.DataFrame(list(Gene_names[0][gene_index]), columns = [0])
peak_name_df = pd.DataFrame(list(Peak_names[0][peak_index]), columns = [0])


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim18_Ordinary11_Rare7"
label.to_csv(os.path.join(workdir, "label.tsv"), sep = "\t", index = False)
cell_name_df.to_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", index = False)
gene_name_df.to_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", index = False)
peak_name_df.to_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", index = False)
mmwrite(os.path.join(workdir, "Gene_Cell.mtx"), RNA_matrix)
mmwrite(os.path.join(workdir, "Peak_Cell.mtx"), ATAC_matrix)


workdir = "./ComicGTN_reproducibility/Data/BMMC-bench-3"
Gene_Peak = sc.read(os.path.join(workdir, "Gene_Peak.mtx"))
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = None)
Gene_Peak.obs_names = Gene_names[0]
Gene_Peak.var_names = Peak_names[0]


workdir = "./ComicGTN_reproducibility/Data/SimulationData/Sim18_Ordinary11_Rare7"
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = 0)
Peak_names = pd.read_csv(os.path.join(workdir, "Peak_names.tsv"), sep = "\t", header = 0)
Gene_Peak1 = Gene_Peak[Gene_names["0"], Peak_names["0"]]
mmwrite(os.path.join(workdir, "Gene_Peak.mtx"), Gene_Peak1.X)


RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
RNA_count = RNA_seq.X
Cell_types = pd.read_csv(os.path.join(workdir, "label.tsv"), sep = "\t", header = 0)
initial_pre = Initial_Clustering(RNA_count) 
cluster_ini_num = len(set(initial_pre)) 
ini_clu = [int(i) for i in initial_pre]
pd.DataFrame(ini_clu).to_csv(os.path.join(workdir, "ini_clu.tsv"), sep = "\t", header = False, index = False)
Cell_types["cell_type"] = Cell_types["cell_type"].map({"CD8+ T": 1, "Naive CD20+ B": 1, "NK": 1, "Transitional B": 1, "CD14+ Mono": 1, 
                                                                                     "CD4+ T activated": 1, "Proerythroblast": 1, "Lymph prog": 1, "B1 B": 1, 
                                                                                     "Erythroblast": 1, "CD4+ T naive": 1,
                                                                                     "CD16+ Mono": 2, "G/M prog": 2, "HSC": 2, "cDC2": 2, "Normoblast": 2, 
                                                                                     "pDC": 2, "Plasma cell": 2})
Cell_types.to_csv(os.path.join(workdir, "FiRE_label.tsv"), sep = "\t", header = False, index = False)